# Self-Improving AI Assistant Workshop

This notebook mirrors the core AMS pipeline in this repo. It now uses a richer profile, a before/after scheduling check, and AMS inspection helpers so the memory lifecycle is easier to see.


## 1) Setup

Make sure Redis and AMS are running first, then execute the cells in order. For the most complete showcase, run `python3 agent.py --showcase` from the CLI.


In [ ]:
import json
import os

import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
AMS_URL = os.environ.get('AMS_URL', 'http://localhost:8000')
AMS_V1_URL = f"{AMS_URL.rstrip('/')}/v1"
USER_ID = os.environ.get('USER_ID', 'test_user')
SESSION_ID = f'ams-demo:{USER_ID}'
MODEL_NAME = os.environ.get('OPENAI_CHAT_MODEL', 'gpt-4o-mini')

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a helpful assistant with access to a user's remembered preferences.
Use remembered facts when they are relevant, but do not mention internal tools or memory systems.
If the user is sharing preferences or personal facts, acknowledge them briefly and respond naturally.
If the user asks for scheduling help and you know a meeting-time preference, use it in your answer.
Do not invent preferences that are not present in the provided memories.
"""

print('AMS_URL =', AMS_URL)
print('USER_ID =', USER_ID)
print('SESSION_ID =', SESSION_ID)
print('OPENAI_API_KEY set =', bool(OPENAI_API_KEY))


## 2) Working memory and long-term memory helpers

These cells use AMS working memory as the primary write path. AMS handles extraction and promotion to long-term memory.


In [ ]:
def healthcheck():
    r = requests.get(f"{AMS_V1_URL}/health")
    r.raise_for_status()
    return r.json()

def get_working_memory(session_id: str, user_id: str):
    r = requests.get(
        f"{AMS_V1_URL}/working-memory/{session_id}",
        params={"user_id": user_id, "model_name": MODEL_NAME},
    )
    if r.status_code == 404:
        return {"session_id": session_id, "user_id": user_id, "messages": [], "memories": [], "context": None}
    r.raise_for_status()
    return r.json()

def put_working_memory(session_id: str, user_id: str, messages: list, context=None, data=None):
    payload = {"user_id": user_id, "messages": messages}
    if context is not None:
        payload["context"] = context
    if data is not None:
        payload["data"] = data
    r = requests.put(
        f"{AMS_V1_URL}/working-memory/{session_id}",
        params={"model_name": MODEL_NAME},
        json=payload,
    )
    r.raise_for_status()
    return r.json()

def search_long_term_memory(user_id: str, query: str, limit: int = 10):
    payload = {"text": query, "user_id": {"eq": user_id}, "limit": limit}
    r = requests.post(f"{AMS_V1_URL}/long-term-memory/search", json=payload)
    r.raise_for_status()
    return r.json()

def list_memories(user_id: str, limit: int = 100):
    payload = {"text": "", "user_id": {"eq": user_id}, "limit": limit}
    r = requests.post(f"{AMS_V1_URL}/long-term-memory/search", json=payload)
    r.raise_for_status()
    return r.json()

def update_memory(memory_id: str, updates: dict):
    r = requests.patch(f"{AMS_V1_URL}/long-term-memory/{memory_id}", json=updates)
    r.raise_for_status()
    return r.json()

def delete_memory(memory_id: str):
    r = requests.delete(f"{AMS_V1_URL}/long-term-memory", params={"memory_ids": memory_id})
    r.raise_for_status()
    return r.json()

healthcheck()


## 3) End-to-end assistant

The notebook now mirrors the CLI: append raw messages to working memory, let AMS extract/promote, then search long-term memory for grounding.


In [ ]:
def recent_transcript(messages, limit=6):
    lines = []
    for message in messages[-limit:]:
        lines.append(f"{message.get('role', 'unknown')}: {message.get('content', '')}")
    return "\n".join(lines)

def fallback_answer(message: str, mem_text: str) -> str:
    lower_message = message.lower()
    if any(phrase in lower_message for phrase in ("schedule", "sync", "calendar", "time slot", "availability")):
        if "afternoon" in mem_text.lower():
            return "You prefer afternoon meetings, so I suggest a 30-minute afternoon slot next week."
        if "morning" in mem_text.lower():
            return "You prefer morning meetings, so I suggest a 30-minute morning slot next week."
        if "evening" in mem_text.lower():
            return "You prefer evening meetings, so I suggest a 30-minute evening slot next week."
        return "I can suggest a 30-minute slot next week. If you share a meeting-time preference, I can tailor it."
    if mem_text:
        return f"I've stored these memories for future turns:\n{mem_text}"
    return "Thanks, I understand. What would you like to do next?"

async def handle_user_message(message: str) -> str:
    working_memory = get_working_memory(SESSION_ID, USER_ID)
    messages = list(working_memory.get("messages", []))
    messages.append({"role": "user", "content": message})
    working_memory = put_working_memory(
        SESSION_ID,
        USER_ID,
        messages,
        context=working_memory.get("context"),
        data=working_memory.get("data"),
    )

    retrieved = search_long_term_memory(USER_ID, message)
    memory_texts = [memory["text"] for memory in retrieved.get("memories", []) if memory.get("text")]
    memory_texts.extend(memory["text"] for memory in working_memory.get("memories", []) if memory.get("text"))
    mem_text = "\n".join(dict.fromkeys(memory_texts))

    user_prompt = (
        "Recent conversation:\n"
        f"{recent_transcript(working_memory.get('messages', [])) or 'None'}\n\n"
        "Relevant memories:\n"
        f"{mem_text or 'None'}\n\n"
        f"Respond to the latest user message: {message}"
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            max_tokens=200,
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception:
        return fallback_answer(message, mem_text)


## 4) Try it out


In [ ]:
sample_messages = [
    'Can you schedule a 30-minute sync next week?',
    "I'm a product engineer based in Seattle. I prefer morning meetings, I'm vegetarian, I'm working on Summit Copilot, and I love hiking and espresso.",
    'Can you schedule a 30-minute sync next week?',
]

sample_messages


## 5) Run the demo

This shows the before/after contrast using AMS-managed extraction from working memory.


In [ ]:
import asyncio

async def run_demo():
    for message in sample_messages:
        print('USER:', message)
        response = await handle_user_message(message)
        print('ASSISTANT:', response)
        print('-' * 80)

# Uncomment to run interactively in Jupyter.
# await run_demo()


## 6) Inspect AMS state

Inspect both working memory and long-term memory after running the demo.


In [ ]:
# Working memory for the active user
# get_working_memory(SESSION_ID, USER_ID)

# Long-term memories for the active user
# list_memories(USER_ID)

# Search a richer profile query
# search_long_term_memory(USER_ID, 'Summit Copilot vegetarian Seattle')


## 7) Manual correction and isolation

Patch a real memory by ID, then ask the scheduling question again. Change `USER_ID` in the setup cell to verify user isolation.


In [ ]:
# Example correction flow. Uncomment to run after inspecting / listing memories.
# memories = list_memories(USER_ID).get("memories", [])
# memory_id = memories[0]["id"] if memories else None
# if memory_id:
#     update_memory(memory_id, {"text": "User prefers afternoon meetings."})
# await handle_user_message("Can you schedule a 30-minute sync next week?")

# To test isolation, set USER_ID = "bob" and SESSION_ID = f"ams-demo:{USER_ID}" in the setup cell, then rerun the profile + query cells.
